# `ar-db-handler` playground

A guided tour through every public helper in `ar_db_handler`.

> **This notebook never touches a real database or a real GCS bucket.**
> Every cell writes to a fresh `tempfile.TemporaryDirectory()` that
> lives only for the lifetime of this kernel — when the kernel shuts
> down, the temp dir is deleted. Your `filings.db`, `metrics.db`, and
> GCS credentials are unaffected.

We'll walk through:

1. Initialising both databases.
2. ID generation — `make_run_id`, `make_file_id`, `EXTENSION_MAP`.
3. Recording a scraper run.
4. Companies + `sync_companies()` with a mocked GCS bridge.
5. The three file statuses — `PENDING` → `FAILED` → `SUCCESS`.
6. `form_type` normalisation, `AlreadyScrapedError`, `force=True`,
   `ValueError` on unknown `file_type`.
7. Amendment handling — 10-K and 10-KA coexist as separate rows.
8. Reading back — `get_file`, `list_companies`, `get_scraped_files`
   (the scraper skip-set), `get_scraped_pairs` (the evaluator entry
   point), `MissingFiscalYearError` for SUCCESS rows without a year.
9. The `scraper_errors` audit table — every rejection is recorded
   before the exception is raised, and `record_error()` for arbitrary
   scraper-side failures.
10. Closing out a run with `update_run_finished`.
11. Writing and reading a `metrics.db` row.
12. Foreign-key enforcement.
13. Teardown.

## 0. Setup — a throwaway working directory

Everything below uses `WORKDIR`, a temporary path. The
`TemporaryDirectory` object is held in a module-level variable so it
isn't garbage-collected (and thus deleted) before we're done with it.

In [1]:
import json
import sqlite3
import sys
import tempfile
import types
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

import ar_db_handler as ardb

# A throwaway directory. Held in a name so it isn't GC'd mid-notebook.
_TMP = tempfile.TemporaryDirectory(prefix="ar_db_playground_")
WORKDIR = Path(_TMP.name)

FILINGS_PATH = WORKDIR / "filings.db"
METRICS_PATH = WORKDIR / "metrics.db"

print(f"ar_db_handler v{ardb.__version__}")
print(f"Working dir : {WORKDIR}")
print(f"filings.db  : {FILINGS_PATH}")
print(f"metrics.db  : {METRICS_PATH}")

ar_db_handler v0.2.0
Working dir : /tmp/ar_db_playground_semdqnhh
filings.db  : /tmp/ar_db_playground_semdqnhh/filings.db
metrics.db  : /tmp/ar_db_playground_semdqnhh/metrics.db


A small helper to ISO-format timestamps the way the SQLite columns store them:

In [2]:
def now_iso() -> str:
    """UTC, ISO-8601, second precision."""
    return datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")


now_iso()

'2026-05-26T09:38:04+00:00'

## 1. Initialise both databases

`init_filings_db(path)` and `init_metrics_db(path)` each return an
open `sqlite3.Connection`. They:

- create the file if it doesn't exist,
- apply the schema (`CREATE TABLE IF NOT EXISTS` everywhere — so
  calling on an existing DB is a safe no-op),
- enable WAL journal mode,
- turn on foreign-key enforcement.

The caller owns the connection and is responsible for closing it. The
notebook keeps both open until the teardown cell.

In [3]:
filings = ardb.init_filings_db(FILINGS_PATH)
metrics = ardb.init_metrics_db(METRICS_PATH)

# Sanity check: the standard pragmas are on.
for name, conn in [("filings", filings), ("metrics", metrics)]:
    jm = conn.execute("PRAGMA journal_mode").fetchone()[0]
    fk = conn.execute("PRAGMA foreign_keys").fetchone()[0]
    print(f"{name:8s}  journal_mode={jm!r}  foreign_keys={fk}")

filings   journal_mode='wal'  foreign_keys=1
metrics   journal_mode='wal'  foreign_keys=1


In [4]:
# Show the tables that were created.
def tables(conn) -> list[str]:
    return [
        r[0]
        for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
    ]


print("filings.db tables:", tables(filings))
print("metrics.db tables:", tables(metrics))

filings.db tables: ['companies', 'files', 'scraper_errors', 'scraper_runs', 'sqlite_sequence']
metrics.db tables: ['metrics']


## 2. ID generation

All IDs flow through `ar_db_handler.ids`. Two patterns:

- **`make_run_id()`** — a UUID4. A scraper run is an ephemeral event
  with no natural business key, so a random ID is the right primitive.
- **`make_file_id(company_id, source_filing_id, file_type)`** — a
  deterministic 16-character SHA-256 prefix derived from the natural
  uniqueness key. The same inputs always produce the same `file_id`,
  which makes the upserts idempotent without a prior SELECT.

`form_type` and `fiscal_year` are NOT part of the hash — they don't
participate in the UNIQUE constraint on the `files` table.

In [5]:
print("make_run_id() — UUID4, one per scraper run:")
print(" ", ardb.make_run_id())
print(" ", ardb.make_run_id())

print("\nmake_file_id() — deterministic 16-char SHA-256 prefix:")
a = ardb.make_file_id(company_id=100, source_filing_id="0000320193-24-000123", file_type="PDF")
b = ardb.make_file_id(company_id=100, source_filing_id="0000320193-24-000123", file_type="PDF")
c = ardb.make_file_id(company_id=100, source_filing_id="0000320193-24-000123", file_type="XBRL")
print(f"  same inputs        → {a}")
print(f"  same inputs again  → {b}    (equal: {a == b})")
print(f"  different file_type → {c}    (different: {a != c})")

print("\nEXTENSION_MAP — file_type → on-disk/GCS extension:")
print(" ", ardb.EXTENSION_MAP)

make_run_id() — UUID4, one per scraper run:
  a7f9dfb8-1557-4f1b-833d-abd65be7335c
  dfec5e18-fd37-45c7-8115-4f5895933988

make_file_id() — deterministic 16-char SHA-256 prefix:
  same inputs        → de61dfb3151abd0b
  same inputs again  → de61dfb3151abd0b    (equal: True)
  different file_type → a2b6fb86d85f9440    (different: True)

EXTENSION_MAP — file_type → on-disk/GCS extension:
  {'PDF': '.pdf', 'XBRL': '.zip'}


## 3. Record a scraper run

A `scraper_runs` row is inserted once at run start with
`status='RUNNING'`. Later, when the run finishes, `update_run_finished`
fills in `finished_at`, `elapsed_time`, and the four count columns.

`upsert_run` uses `INSERT OR IGNORE`: re-issuing the same `scraper_id`
silently does nothing, so a crash-and-resume can't accidentally
clobber the original start time.

In [6]:
scraper_id = ardb.make_run_id()

ardb.upsert_run(
    filings,
    ardb.RunRecord(
        scraper_id=scraper_id,
        country_code="US",
        workers_count=3,
        source_file="scripts/scrape_us.py",
        log_path="/tmp/scrape_us.log",
        version="1.0.0",
        started_at=now_iso(),
        status="RUNNING",
        metadata=json.dumps({"source": "EDGAR"}),
    ),
)

row = filings.execute(
    "SELECT scraper_id, country_code, status, workers_count "
    "FROM scraper_runs WHERE scraper_id = ?",
    (scraper_id,),
).fetchone()
print(f"scraper_id   : {row[0]}")
print(f"country_code : {row[1]}")
print(f"status       : {row[2]}")
print(f"workers      : {row[3]}")

scraper_id   : da02c82a-fdc8-48ad-be6b-61969fa161e7
country_code : US
status       : RUNNING
workers      : 3


**Try to re-upsert with a different status** — the original `RUNNING`
row must survive, because `upsert_run` is `INSERT OR IGNORE`:

In [7]:
ardb.upsert_run(
    filings,
    ardb.RunRecord(
        scraper_id=scraper_id,
        country_code="US",
        workers_count=99,          # would-be update
        source_file=None,
        log_path=None,
        version=None,
        started_at=now_iso(),
        status="SUCCESS",          # would-be update
        metadata=None,
    ),
)

row = filings.execute(
    "SELECT status, workers_count FROM scraper_runs WHERE scraper_id = ?",
    (scraper_id,),
).fetchone()
print(f"status after re-upsert  : {row[0]}")
print(f"workers after re-upsert : {row[1]}")
print("→ original row preserved")

status after re-upsert  : RUNNING
workers after re-upsert : 3
→ original row preserved


## 4. Companies and `sync_companies()`

`upsert_company` is `INSERT OR REPLACE` and forces
`is_in_company_info = 1` plus a fresh `last_synced_at` on every call.
This is the happy-path of `sync_companies()`; companies absent from
the master snapshot are flipped to `is_in_company_info = 0` in a
separate sweep first.

We'll seed three companies by hand here, then demonstrate
`sync_companies()` with a mocked GCS bridge.

In [8]:
for cid, ticker, cc, country in [
    (100, "ACME", "US", "United States"),
    (200, "BETA", "US", "United States"),
    (300, "TYO",  "JP", "Japan"),
]:
    ardb.upsert_company(
        filings,
        ardb.CompanyRecord(
            company_id=cid,
            fs_ticker=ticker,
            country_code=cc,
            country=country,
            country_id=cc,
            file_name=ticker.lower(),
            coverage_status="LAFA",
            start_year_force=2008,
        ),
    )

print("Companies (using list_companies):")
for c in ardb.list_companies(filings):
    print(
        f"  {c['company_id']:>4d}  {c['fs_ticker']:6s}  {c['country_code']:2s}  "
        f"active={c['is_in_company_info']}  last_synced={c['last_synced_at']}"
    )

Companies (using list_companies):
   100  ACME    US  active=1  last_synced=2026-05-26T09:38:04+00:00
   200  BETA    US  active=1  last_synced=2026-05-26T09:38:04+00:00
   300  TYO     JP  active=1  last_synced=2026-05-26T09:38:04+00:00


### `sync_companies()` — with a fake bridge

`sync_companies()` reaches out to GCS via `GCPWeeklyFiles` and:

1. Calls `get_latest_period()` to find the most recent snapshot.
2. Reads `company_info.parquet` from that period.
3. Filters to `country_code` if one is passed.
4. Sets `is_in_company_info = 0` for every existing row in scope
   **before** upserting (so anything absent from the snapshot ends
   up deactivated automatically).
5. Upserts each row from the snapshot — which re-activates it.
6. Returns a `SyncResult(period, upserted, delisted, country_code)`.

For this notebook we install a fake `gcpBridge` module into
`sys.modules` so we can demonstrate the behaviour without real
credentials. The snapshot we hand it removes BETA (which was active)
and adds a new company DELT.

In [9]:
class _FakeGCPWeeklyFiles:
    def __init__(self, credentials_path=None):
        print(f"  [fake bridge: credentials_path={credentials_path!r}]")

    def get_latest_period(self):
        return "2026-05-22-W3"

    def read_file_from_period(self, period, filename):
        print(f"  [fake bridge: read {filename} from period {period}]")
        # ACME survives (with an updated file_name), BETA is gone,
        # DELT is new. JP is not in the snapshot at all.
        return pd.DataFrame(
            [
                {
                    "company_id": 100,
                    "fs_ticker": "ACME",
                    "country_code": "US",
                    "country": "United States",
                    "country_id": "US",
                    "file_name": "acme_v2",       # ← updated
                    "coverage_status": "LAFA",
                    "start_year_force": 2008,
                },
                {
                    "company_id": 400,
                    "fs_ticker": "DELT",
                    "country_code": "US",
                    "country": "United States",
                    "country_id": "US",
                    "file_name": "delt",
                    "coverage_status": "LAFA",
                    "start_year_force": 2015,
                },
            ]
        )


fake_mod = types.ModuleType("gcpBridge")
fake_mod.GCPWeeklyFiles = _FakeGCPWeeklyFiles
sys.modules["gcpBridge"] = fake_mod

result = ardb.sync_companies(filings, country_code="US")
print(f"\nresult: {result}")

  [fake bridge: credentials_path=None]
  [fake bridge: read company_info.parquet from period 2026-05-22-W3]

result: SyncResult(period='2026-05-22-W3', upserted=2, delisted=1, country_code='US')


In [10]:
print("US companies after sync:")
for c in ardb.list_companies(filings, country_code="US", active_only=False):
    flag = "active" if c["is_in_company_info"] == 1 else "DELISTED"
    print(f"  {c['company_id']:>4d}  {c['fs_ticker']:6s}  {flag:8s}  {c['file_name']}")

# Important — the JP row is OUT of scope of a US sync and must NOT be touched.
jp = ardb.list_companies(filings, country_code="JP", active_only=False)
print(
    f"\nJP rows untouched by US-scoped sync: "
    f"{len(jp)} row(s), still active={jp[0]['is_in_company_info']}"
)

US companies after sync:
   100  ACME    active    acme_v2
   200  BETA    DELISTED  beta
   400  DELT    active    delt

JP rows untouched by US-scoped sync: 1 row(s), still active=1


## 5. File lifecycle — `PENDING` → `FAILED` → `SUCCESS`

A `files` row has a `status` of `PENDING`, `FAILED`, or `SUCCESS`.
The natural key is `(company_id, source_filing_id, file_type)`.

`upsert_file` derives three things automatically — the caller never
sets them:

- `file_id`    — `make_file_id(company_id, source_filing_id, file_type)`
- `extension`  — `EXTENSION_MAP[file_type]` (raises `ValueError` on
  unknown `file_type`)
- `form_type`  — `None`, `""`, or whitespace is normalised to `"UNKNOWN"`

We'll walk through a realistic lifecycle:

1. Scraper records `PENDING` before attempting download.
2. Download fails — row is updated to `FAILED`.
3. Retry succeeds — row becomes `SUCCESS`.

Each step calls `upsert_file`. As long as the existing row is not yet
`SUCCESS`, the helper just does an `INSERT OR REPLACE`.

In [11]:
ACME_2023_PDF = ardb.make_file_id(100, "0000320193-24-acme-2023", "PDF")
print(f"file_id (deterministic): {ACME_2023_PDF}")


def file_record(
    *, company_id, source_filing_id, file_type, status,
    form_type="10-K", gcs_path=None, error_message=None,
    fiscal_year=2023, reporting_date="2023-12-31",
):
    """Convenience builder so the cells below stay short."""
    return ardb.FileRecord(
        company_id=company_id,
        scraper_id=scraper_id,
        status=status,
        country_code="US",
        file_type=file_type,
        source_filing_id=source_filing_id,
        form_type=form_type,
        fiscal_year=fiscal_year,
        reporting_date=reporting_date,
        filing_date="2024-02-15",
        gcs_path=gcs_path,
        url="https://www.sec.gov/Archives/edgar/data/...",
        scraped_at=now_iso() if status == "SUCCESS" else None,
        error_message=error_message,
    )


# Step 1 — intent recorded before download.
ardb.upsert_file(
    filings,
    file_record(
        company_id=100,
        source_filing_id="0000320193-24-acme-2023",
        file_type="PDF",
        status="PENDING",
    ),
)
print("step 1 status:", ardb.get_file(filings, ACME_2023_PDF)["status"])

file_id (deterministic): 4ce3250428130d8d
step 1 status: PENDING


In [12]:
# Step 2 — download failed, mark FAILED.
ardb.upsert_file(
    filings,
    file_record(
        company_id=100,
        source_filing_id="0000320193-24-acme-2023",
        file_type="PDF",
        status="FAILED",
        error_message="404 from EDGAR",
    ),
)
row = ardb.get_file(filings, ACME_2023_PDF)
print(f"step 2 status        : {row['status']}")
print(f"step 2 error_message : {row['error_message']}")

step 2 status        : FAILED
step 2 error_message : 404 from EDGAR


In [13]:
# Step 3 — retry succeeded. FAILED → SUCCESS is allowed WITHOUT force.
ardb.upsert_file(
    filings,
    file_record(
        company_id=100,
        source_filing_id="0000320193-24-acme-2023",
        file_type="PDF",
        status="SUCCESS",
        gcs_path="gs://my-bucket/100/100_10K_2023-12-31.pdf",
    ),
)
row = ardb.get_file(filings, ACME_2023_PDF)
print(f"step 3 status    : {row['status']}")
print(f"step 3 gcs_path  : {row['gcs_path']}")
print(f"step 3 extension : {row['extension']}    ← auto-derived from file_type='PDF'")
print(f"step 3 form_type : {row['form_type']}")

step 3 status    : SUCCESS
step 3 gcs_path  : gs://my-bucket/100/100_10K_2023-12-31.pdf
step 3 extension : .pdf    ← auto-derived from file_type='PDF'
step 3 form_type : 10-K


### `form_type` normalisation

The scraper sometimes can't determine the form type. Whatever the
caller hands in — `None`, the empty string, or just whitespace — is
normalised to `"UNKNOWN"` inside `upsert_file`. The
`form_type NOT NULL DEFAULT 'UNKNOWN'` column constraint catches any
caller path that bypasses the helper.

In [14]:
weird_id = ardb.make_file_id(100, "test-norm", "PDF")
ardb.upsert_file(
    filings,
    file_record(
        company_id=100,
        source_filing_id="test-norm",
        file_type="PDF",
        status="SUCCESS",
        form_type=None,             # ← caller doesn't know
        gcs_path="gs://my-bucket/test.pdf",
    ),
)
ft = ardb.get_file(filings, weird_id)["form_type"]
print(f"None / empty / whitespace form_type → stored as {ft!r}")

None / empty / whitespace form_type → stored as 'UNKNOWN'


### `AlreadyScrapedError` — the SUCCESS row is now sticky

Once a row is `SUCCESS`, `upsert_file` raises `AlreadyScrapedError`
rather than overwriting it. The scraper's primary defence is
`get_scraped_files()` (a fast O(1) skip-set, used at startup); the
exception is the secondary guard, mainly useful when multiple
workers race on the same filing.

In [15]:
try:
    ardb.upsert_file(
        filings,
        file_record(
            company_id=100,
            source_filing_id="0000320193-24-acme-2023",
            file_type="PDF",
            status="SUCCESS",
            gcs_path="gs://my-bucket/100/replacement.pdf",
        ),
    )
except ardb.AlreadyScrapedError as e:
    print("caught AlreadyScrapedError:")
    print(f"  {e}")

caught AlreadyScrapedError:
  File already scraped (status=SUCCESS) for company_id=100, source_filing_id='0000320193-24-acme-2023', file_type='PDF'. Pass force=True to overwrite.


### `force=True` — explicit re-scrape

When you really do want to overwrite a `SUCCESS` row (e.g. the GCS
file was deleted, or you're refreshing), pass `force=True`.

In [16]:
ardb.upsert_file(
    filings,
    file_record(
        company_id=100,
        source_filing_id="0000320193-24-acme-2023",
        file_type="PDF",
        status="SUCCESS",
        gcs_path="gs://my-bucket/100/refreshed.pdf",
    ),
    force=True,
)
row = ardb.get_file(filings, ACME_2023_PDF)
print(f"gcs_path after force=True : {row['gcs_path']}")

gcs_path after force=True : gs://my-bucket/100/refreshed.pdf


### `ValueError` on unknown `file_type`

The cheapest validation in `upsert_file` is the `EXTENSION_MAP`
lookup. An unknown `file_type` raises `ValueError` **before** any
INSERT — no orphan row ends up in the database.

In [17]:
try:
    ardb.upsert_file(
        filings,
        ardb.FileRecord(
            company_id=100,
            scraper_id=scraper_id,
            status="SUCCESS",
            country_code="US",
            file_type="HTML",           # ← not in EXTENSION_MAP
            source_filing_id="test-bad",
            form_type="10-K",
            fiscal_year=2024,
            reporting_date="2024-12-31",
            filing_date="2025-02-01",
            gcs_path="gs://bucket/x",
            url=None,
            scraped_at=None,
            error_message=None,
        ),
    )
except ValueError as e:
    print(f"caught ValueError:\n  {e}")

caught ValueError:
  Unknown file_type 'HTML'. Expected one of: PDF, XBRL.


## 6. Build out a realistic table to exercise the read helpers

We want this final shape:

| company | filing              | PDF      | XBRL     | Pair-able? |
|---------|---------------------|----------|----------|------------|
| ACME    | 2023                | SUCCESS  | SUCCESS  | yes        |
| ACME    | 2024 (10-K)         | SUCCESS  | PENDING  | no         |
| ACME    | 2024 (10-KA amend.) | SUCCESS  | SUCCESS  | yes        |
| BETA    | 2024                | FAILED   | SUCCESS  | no         |

The 10-K and 10-KA for ACME 2024 share `(company_id, fiscal_year)`
but have **different `source_filing_id`s**, so they coexist as two
separate rows — exactly what the spec calls amendment handling.

In [18]:
# Drop the form-norm test row first so it doesn't pollute the pair demo.
filings.execute("DELETE FROM files WHERE source_filing_id = 'test-norm'")
filings.commit()


def add(company_id, sfid, file_type, status, fy, form_type="10-K"):
    ardb.upsert_file(
        filings,
        file_record(
            company_id=company_id,
            source_filing_id=sfid,
            file_type=file_type,
            status=status,
            form_type=form_type,
            fiscal_year=fy,
            reporting_date=f"{fy}-12-31" if fy is not None else None,
            gcs_path=(
                f"gs://my-bucket/{company_id}/{company_id}_"
                f"{form_type.replace('-', '')}_{fy}."
                f"{'zip' if file_type == 'XBRL' else 'pdf'}"
                if status == "SUCCESS" else None
            ),
        ),
    )


# ACME 2023 XBRL (PDF already SUCCESS from earlier cells).
add(100, "0000320193-24-acme-2023", "XBRL", "SUCCESS", fy=2023)

# ACME 2024 10-K — PDF SUCCESS, XBRL PENDING (the gap).
add(100, "0000320193-25-acme-2024",       "PDF",  "SUCCESS", fy=2024, form_type="10-K")
add(100, "0000320193-25-acme-2024",       "XBRL", "PENDING", fy=2024, form_type="10-K")

# ACME 2024 10-KA amendment — both SUCCESS.
add(100, "0000320193-25-acme-2024-amend", "PDF",  "SUCCESS", fy=2024, form_type="10-KA")
add(100, "0000320193-25-acme-2024-amend", "XBRL", "SUCCESS", fy=2024, form_type="10-KA")

# BETA 2024 — PDF FAILED, XBRL SUCCESS.
add(200, "0000320193-25-beta-2024", "PDF",  "FAILED",  fy=2024)
add(200, "0000320193-25-beta-2024", "XBRL", "SUCCESS", fy=2024)

print(f"{'company':>7s}  {'source_filing_id':<40s}  {'type':<5s}  {'form':<7s}  {'fy':>4s}  status")
print("-" * 90)
for row in filings.execute(
    "SELECT company_id, source_filing_id, file_type, form_type, fiscal_year, status "
    "FROM files ORDER BY company_id, source_filing_id, file_type"
):
    fy_str = str(row[4]) if row[4] is not None else "----"
    print(
        f"{row[0]:>7d}  {row[1]:<40s}  {row[2]:<5s}  {row[3]:<7s}  "
        f"{fy_str:>4s}  {row[5]}"
    )

company  source_filing_id                          type   form       fy  status
------------------------------------------------------------------------------------------
    100  0000320193-24-acme-2023                   PDF    10-K     2023  SUCCESS
    100  0000320193-24-acme-2023                   XBRL   10-K     2023  SUCCESS
    100  0000320193-25-acme-2024                   PDF    10-K     2024  SUCCESS
    100  0000320193-25-acme-2024                   XBRL   10-K     2024  PENDING
    100  0000320193-25-acme-2024-amend             PDF    10-KA    2024  SUCCESS
    100  0000320193-25-acme-2024-amend             XBRL   10-KA    2024  SUCCESS
    200  0000320193-25-beta-2024                   PDF    10-K     2024  FAILED
    200  0000320193-25-beta-2024                   XBRL   10-K     2024  SUCCESS


## 7. `get_scraped_files()` — the scraper skip-set

Called once per scraper run (or per worker) at startup. Returns a
`set` of `(company_id, source_filing_id, file_type)` tuples for every
`SUCCESS` row. Membership check is O(1).

`fiscal_year` is deliberately **not** in the tuple. It's derived
metadata and can be `NULL` for unresolvable filings — using it as the
skip anchor would cause false misses. `source_filing_id` (the
regulator-assigned ID) is the right anchor: always present, always
unique, never derived.

In [19]:
skip = ardb.get_scraped_files(filings, country_code="US")
print(f"skip-set has {len(skip)} entries (SUCCESS rows only):")
for entry in sorted(skip):
    print(f"  {entry}")

# Real-world usage pattern:
candidate = (100, "0000320193-24-acme-2023", "PDF")
print(f"\nwould we re-scrape {candidate}?")
print(f"  → {candidate not in skip}")

skip-set has 6 entries (SUCCESS rows only):
  (100, '0000320193-24-acme-2023', 'PDF')
  (100, '0000320193-24-acme-2023', 'XBRL')
  (100, '0000320193-25-acme-2024', 'PDF')
  (100, '0000320193-25-acme-2024-amend', 'PDF')
  (100, '0000320193-25-acme-2024-amend', 'XBRL')
  (200, '0000320193-25-beta-2024', 'XBRL')

would we re-scrape (100, '0000320193-24-acme-2023', 'PDF')?
  → False


## 8. `get_scraped_pairs()` — the evaluator entry point

Returns every filing for which **both** the PDF and the XBRL are
`SUCCESS` — i.e. ready to evaluate. Implemented as a self-join on
`files` so anything missing one side is dropped at the SQL level —
no Python-side filtering.

**Important**: `ar-db-handler` is country-agnostic. When multiple
form types exist for the same `(company_id, fiscal_year, file_type)`
— e.g. a 10-K and a 10-KA — `get_scraped_pairs` returns **all
combinations**. Picking the preferred one is the caller's job
(country-specific scraper or evaluator).

Given our table above, ACME 2024 has:
- 1 10-K PDF + 1 10-KA PDF
- the 10-K XBRL is still PENDING; only the 10-KA XBRL is SUCCESS

That's 2 PDFs × 1 XBRL = 2 pairs for ACME 2024. Plus 1 pair for
ACME 2023 = 3 pairs total. BETA 2024 produces nothing (PDF FAILED).

In [20]:
pairs = ardb.get_scraped_pairs(filings)
print(f"{len(pairs)} pair(s) ready for evaluation:\n")
for p in pairs:
    print(f"  company {p.company_id}, FY{p.fiscal_year}")
    print(f"    PDF  ({p.pdf_form_type:6s}): {p.pdf_gcs_path}")
    print(f"    XBRL ({p.xbrl_form_type:6s}): {p.xbrl_gcs_path}")
    print()

3 pair(s) ready for evaluation:

  company 100, FY2023
    PDF  (10-K  ): gs://my-bucket/100/refreshed.pdf
    XBRL (10-K  ): gs://my-bucket/100/100_10K_2023.zip

  company 100, FY2024
    PDF  (10-K  ): gs://my-bucket/100/100_10K_2024.pdf
    XBRL (10-KA ): gs://my-bucket/100/100_10KA_2024.zip

  company 100, FY2024
    PDF  (10-KA ): gs://my-bucket/100/100_10KA_2024.pdf
    XBRL (10-KA ): gs://my-bucket/100/100_10KA_2024.zip



**Optional filters** — `company_id` and `fiscal_year` both AND together:

In [21]:
acme_2024 = ardb.get_scraped_pairs(filings, company_id=100, fiscal_year=2024)
print(f"ACME FY2024 pairs (no priority applied — caller chooses): {len(acme_2024)}")
for p in acme_2024:
    print(f"  PDF {p.pdf_form_type:6s} × XBRL {p.xbrl_form_type:6s}")

ACME FY2024 pairs (no priority applied — caller chooses): 2
  PDF 10-K   × XBRL 10-KA 
  PDF 10-KA  × XBRL 10-KA 


### Rows with `fiscal_year IS NULL`

Under the v0.2 invariant, **a SUCCESS row MUST carry a non-null
`fiscal_year`** (Python check + SQL CHECK constraint). The
`reporting_date` is always available from the source API
(EDGAR `reportDate`, EDINET `periodEnd`, ...), so a successful
scrape with no derivable year indicates a scraper bug, not a data
limitation.

NULL `fiscal_year` IS still allowed on `PENDING` and `FAILED` rows
— they don't represent committed work. `get_scraped_pairs` excludes
them via `WHERE fiscal_year IS NOT NULL`, so they don't produce
spurious matches.

In [22]:
# A PENDING row with NULL fiscal_year is fine: the scraper hasn't
# fetched metadata yet, so it doesn't know the year.
add(300, "S100PENDING", "PDF",  "PENDING", fy=None)
# The helper hard-codes country_code='US'; fix that for the JP row.
filings.execute("UPDATE files SET country_code='JP' WHERE company_id=300")
filings.commit()

still_pairs = ardb.get_scraped_pairs(filings, company_id=300)
print(f"JP company with PENDING+NULL fiscal_year → pairs returned: {len(still_pairs)}")
print("  (correctly excluded — PENDING rows aren't pair-eligible)")

# The skip-set is keyed on SUCCESS rows only, so PENDING rows don't appear here.
# The scraper uses get_scraped_files to skip COMMITTED work; in-flight rows
# are handled separately.
jp_skip = ardb.get_scraped_files(filings, country_code="JP")
print(f"  skip-set sees only SUCCESS rows: {len(jp_skip)} entries")

JP company with PENDING+NULL fiscal_year → pairs returned: 0
  (correctly excluded — PENDING rows aren't pair-eligible)
  skip-set sees only SUCCESS rows: 0 entries


### `MissingFiscalYearError` — SUCCESS without `fiscal_year` is rejected

The Python guard fires before the INSERT, so no half-baked row ends
up in the table. The rejection is also recorded to `scraper_errors`
before the exception is raised — we'll inspect the audit table in the
next section.

In [23]:
try:
    ardb.upsert_file(
        filings,
        file_record(
            company_id=300,
            source_filing_id="S100BUG",
            file_type="PDF",
            status="SUCCESS",       # SUCCESS + ...
            fiscal_year=None,       # ...no year → must be rejected
            gcs_path="gs://my-bucket/300/buggy.pdf",
        ),
    )
except ardb.MissingFiscalYearError as e:
    print("caught MissingFiscalYearError:")
    print(f"  {e}")

caught MissingFiscalYearError:
  upsert_file: status='SUCCESS' requires fiscal_year != None. company_id=300, source_filing_id='S100BUG', file_type='PDF'. Derive fiscal_year from reporting_date in the scraper before calling.


## 9. The `scraper_errors` audit table

Every rejection we just walked through — `ValueError` on unknown
`file_type`, `MissingFiscalYearError`, `AlreadyScrapedError`, and the
FK violation later — **also wrote a row** to a dedicated
`scraper_errors` table BEFORE the exception was raised.

This means the audit trail is durable even if the caller swallows the
exception. The `get_scraper_errors()` helper queries it.

In [24]:
# Everything recorded for THIS run.
errs = ardb.get_scraper_errors(filings, scraper_id=scraper_id)

print(f"Errors recorded so far in run {scraper_id[:8]}…: {len(errs)}\n")
print(f"{'#':>2s}  {'error_type':<25s}  {'company':>7s}  source_filing_id")
print("-" * 80)
for i, e in enumerate(errs, 1):
    sfid = e["source_filing_id"] or "(none)"
    cid = e["company_id"] if e["company_id"] is not None else "—"
    print(f"{i:>2d}  {e['error_type']:<25s}  {str(cid):>7s}  {sfid}")

Errors recorded so far in run da02c82a…: 3

 #  error_type                 company  source_filing_id
--------------------------------------------------------------------------------
 1  MISSING_FISCAL_YEAR            300  S100BUG
 2  UNKNOWN_FILE_TYPE              100  test-bad
 3  ALREADY_SCRAPED                100  0000320193-24-acme-2023


**Filter by error type** — useful when you want to see only one
category. The `ERROR_*` string constants are re-exported from
`ar_db_handler` so SQL filters and Python `except` clauses can't
drift apart:

In [25]:
already = ardb.get_scraper_errors(filings, error_type=ardb.ERROR_ALREADY_SCRAPED)
print(f"ALREADY_SCRAPED events: {len(already)}")
for e in already:
    print(f"  {e['source_filing_id']!r} ({e['file_type']})")

print()

missing_fy = ardb.get_scraper_errors(filings, error_type=ardb.ERROR_MISSING_FISCAL_YEAR)
print(f"MISSING_FISCAL_YEAR events: {len(missing_fy)}")
for e in missing_fy:
    print(f"  {e['source_filing_id']!r}: {e['error_message'][:80]}...")

ALREADY_SCRAPED events: 1
  '0000320193-24-acme-2023' (PDF)

MISSING_FISCAL_YEAR events: 1
  'S100BUG': upsert_file: status='SUCCESS' requires fiscal_year != None. company_id=300, sour...


### Recording your own errors

The scraper can call `record_error()` directly for anything that
didn't go through one of the helpers — a download timeout, an HTML
parse failure, a GCS upload error. The `error_type` is a free-form
string when the caller defines it (the package-defined constants are
just conventions).

In [26]:
ardb.record_error(
    filings,
    ardb.ErrorRecord(
        scraper_id=scraper_id,
        error_type="DOWNLOAD_TIMEOUT",          # caller-defined; not in ERROR_*
        error_message="EDGAR timed out after 30s on 3 retries",
        company_id=100,
        source_filing_id="0000320193-25-timeout",
        file_type="PDF",
        payload=json.dumps({"url": "https://www.sec.gov/...", "attempts": 3}),
    ),
)

# And we can filter by our custom type too.
timeouts = ardb.get_scraper_errors(filings, error_type="DOWNLOAD_TIMEOUT")
print(f"DOWNLOAD_TIMEOUT events: {len(timeouts)}")
for e in timeouts:
    print(f"  {e['error_message']}")
    print(f"  payload: {e['payload']}")

DOWNLOAD_TIMEOUT events: 1
  EDGAR timed out after 30s on 3 retries
  payload: {"url": "https://www.sec.gov/...", "attempts": 3}


### System-level errors

`sync_companies()` records failures under
`scraper_id = SYSTEM_SCRAPER_ID` (literal string `"SYSTEM"`) — they
don't belong to any scraper run. The two categories are
`ERROR_SYNC_NO_PERIOD` and `ERROR_SNAPSHOT_SCHEMA_DRIFT`. None of
those have fired in this notebook (sync was happy), so the system
view is empty:

In [27]:
system_errs = ardb.get_scraper_errors(filings, scraper_id=ardb.SYSTEM_SCRAPER_ID)
print(f"System-level errors: {len(system_errs)}")

System-level errors: 0


## 10. Close the run out with `update_run_finished`

At run end, the scraper calls `update_run_finished` to record the
final status and the four count columns.

In [28]:
n_success = filings.execute(
    "SELECT COUNT(*) FROM files WHERE status='SUCCESS' AND scraper_id=?",
    (scraper_id,),
).fetchone()[0]
n_xbrl = filings.execute(
    "SELECT COUNT(*) FROM files WHERE status='SUCCESS' AND file_type='XBRL' AND scraper_id=?",
    (scraper_id,),
).fetchone()[0]
n_pdf = filings.execute(
    "SELECT COUNT(*) FROM files WHERE status='SUCCESS' AND file_type='PDF' AND scraper_id=?",
    (scraper_id,),
).fetchone()[0]
n_fail = filings.execute(
    "SELECT COUNT(*) FROM files WHERE status='FAILED' AND scraper_id=?",
    (scraper_id,),
).fetchone()[0]

ardb.update_run_finished(
    filings,
    scraper_id=scraper_id,
    status="SUCCESS",
    finished_at=now_iso(),
    elapsed_time=12.3,
    scraped_files=n_success,
    xbrl_count=n_xbrl,
    pdf_count=n_pdf,
    fail_count=n_fail,
)

closing = filings.execute(
    "SELECT status, finished_at, elapsed_time, scraped_files, xbrl_count, pdf_count, fail_count "
    "FROM scraper_runs WHERE scraper_id = ?",
    (scraper_id,),
).fetchone()
print("scraper_runs after close-out:")
print(f"  status        : {closing[0]}")
print(f"  finished_at   : {closing[1]}")
print(f"  elapsed_time  : {closing[2]}s")
print(f"  scraped_files : {closing[3]}")
print(f"  xbrl / pdf    : {closing[4]} / {closing[5]}")
print(f"  fail_count    : {closing[6]}")

scraper_runs after close-out:
  status        : SUCCESS
  finished_at   : 2026-05-26T09:38:04+00:00
  elapsed_time  : 12.3s
  scraped_files : 6
  xbrl / pdf    : 3 / 3
  fail_count    : 1


## 11. `metrics.db` — write_metric / get_metric (stub)

The `metrics` table is a stub today — only `evaluation_id` (PK) and
`file_id` (cross-DB reference to `filings.db`) are defined. The full
column set will be added once the evaluator schema is finalised.

`file_id` is **not** a SQL foreign key — SQLite can't enforce FKs
across separate `.db` files, and a hard coupling between the two
databases is the wrong default. Callers reconcile by `file_id` at
query time.

The helper accepts arbitrary `**columns` kwargs so it'll keep
working once new columns are added to `schema.sql`.

In [29]:
eid = ardb.write_metric(metrics, file_id=ACME_2023_PDF)
print(f"wrote evaluation : {eid}")

row = ardb.get_metric(metrics, eid)
print(f"read back        : {row}")

wrote evaluation : ab6740c9-f8fa-42df-b142-f2b940f8709d
read back        : {'evaluation_id': 'ab6740c9-f8fa-42df-b142-f2b940f8709d', 'file_id': '4ce3250428130d8d'}


## 12. Foreign-key enforcement

FK pragma is on, so writing a `files` row that points at a
non-existent `company_id` is rejected at insert time.

In [30]:
try:
    ardb.upsert_file(
        filings,
        ardb.FileRecord(
            company_id=99999,           # ← no such company
            scraper_id=scraper_id,
            status="PENDING",
            country_code="US",
            file_type="PDF",
            source_filing_id="bad-fk-test",
            form_type="10-K",
            fiscal_year=2024,
            reporting_date="2024-12-31",
            filing_date="2025-02-01",
            gcs_path=None,
            url=None,
            scraped_at=None,
            error_message=None,
        ),
    )
except sqlite3.IntegrityError as e:
    print(f"caught IntegrityError on bogus company_id:\n  {e}")

# Critically: the rejection was recorded to scraper_errors before the
# IntegrityError was re-raised. The audit row survives even though the
# files insert did not.
print()
errs = ardb.get_scraper_errors(filings, error_type=ardb.ERROR_FK_VIOLATION)
print(f"FK violations recorded: {len(errs)}")
print(f"  most recent: company_id={errs[0]['company_id']}, "
      f"source_filing_id={errs[0]['source_filing_id']!r}")

caught IntegrityError on bogus company_id:
  FOREIGN KEY constraint failed

FK violations recorded: 1
  most recent: company_id=99999, source_filing_id='bad-fk-test'


## 13. Teardown

Close both connections. Once this notebook's kernel shuts down, the
`TemporaryDirectory` is destroyed and both databases vanish with it
— nothing else on disk was touched.

In [31]:
filings.close()
metrics.close()

print("Files still present in temp dir (will be deleted on kernel shutdown):")
for p in sorted(WORKDIR.iterdir()):
    print(f"  {p.name:20s}  {p.stat().st_size:>8d} bytes")

Files still present in temp dir (will be deleted on kernel shutdown):
  filings.db               69632 bytes
  metrics.db               12288 bytes


---

### Cheatsheet — upsert behaviour

| Helper                  | Conflict key                                  | Behaviour                                                                       |
|-------------------------|-----------------------------------------------|---------------------------------------------------------------------------------|
| `upsert_run`            | `scraper_id`                                  | `INSERT OR IGNORE` — first row wins                                             |
| `update_run_finished`   | `scraper_id`                                  | `UPDATE` — sets finished_at, elapsed_time, status, and the four count columns   |
| `upsert_company`        | `company_id`                                  | `INSERT OR REPLACE` — always sets `is_in_company_info=1` and `last_synced_at`   |
| `upsert_file`           | `(company_id, source_filing_id, file_type)`   | replace if not yet SUCCESS; else raise `AlreadyScrapedError` (or replace with `force=True`) |
| `write_metric`          | `evaluation_id`                               | `INSERT` (auto-UUID by default)                                                 |

### Cheatsheet — key derivation rules

| Field         | Set by              | Rule                                                                              |
|---------------|---------------------|-----------------------------------------------------------------------------------|
| `file_id`     | `upsert_file`       | `make_file_id(company_id, source_filing_id, file_type)` — deterministic           |
| `extension`   | `upsert_file`       | `EXTENSION_MAP[file_type]` — raises `ValueError` on unknown                       |
| `form_type`   | `upsert_file`       | `None` / `""` / whitespace → `"UNKNOWN"`                                          |
| `fiscal_year` | scraper             | `year(reporting_date)` — MUST be set when `status='SUCCESS'`                      |

### Cheatsheet — auto-recorded errors

Every rejection from `upsert_file` and `sync_companies` writes a row
to `scraper_errors` **before** the exception propagates. Use
`get_scraper_errors()` to inspect.

| Trigger                                | `error_type` constant            | Python exception            | scraper_id      |
|----------------------------------------|----------------------------------|-----------------------------|-----------------|
| unknown `file_type`                    | `ERROR_UNKNOWN_FILE_TYPE`        | `ValueError`                | run UUID        |
| `status='SUCCESS'` & `fiscal_year=None`| `ERROR_MISSING_FISCAL_YEAR`      | `MissingFiscalYearError`    | run UUID        |
| SUCCESS row exists, `force=False`      | `ERROR_ALREADY_SCRAPED`          | `AlreadyScrapedError`       | run UUID        |
| bogus `company_id` / `scraper_id`      | `ERROR_FK_VIOLATION`             | `sqlite3.IntegrityError`    | run UUID        |
| other CHECK constraint failure         | `ERROR_CHECK_VIOLATION`          | `sqlite3.IntegrityError`    | run UUID        |
| `get_latest_period()` returned None    | `ERROR_SYNC_NO_PERIOD`           | `RuntimeError`              | `SYSTEM`        |
| snapshot row missing required column   | `ERROR_SNAPSHOT_SCHEMA_DRIFT`    | `KeyError`                  | `SYSTEM`        |

Plus: the scraper can call `record_error()` directly for any
download/parse failure not surfaced by the helpers above.